In [8]:
import pandas as pd
from pathlib import Path
import re
from difflib import SequenceMatcher
from unidecode import unidecode
from scipy.optimize import linear_sum_assignment
import numpy as np

MAIN_FOLDER = r"C:\Users\L14\Downloads\ABH_CSV"

#path = r"C:\Users\L14\Documents\SITUATION GLOBALE\CONTROLE_02_02_2026\CONTROLE CTB REMARQUES_total_fusionne.xlsx"

path_maj_gestionnaire = fr"{MAIN_FOLDER}\DEMANDES_total_fusionne.xlsx"
path_voisins_pvcl = fr"{MAIN_FOLDER}\VOISINS_PVCL_total_fusionne.xlsx"
path_voisins_signes = fr"{MAIN_FOLDER}\VOISINS_PRESENCE_total_fusionne.xlsx"
path_voisins_non_par = fr"{MAIN_FOLDER}\VOISINS_NON_PAR_total_fusionne.xlsx"
path_etat_voisin = fr"{MAIN_FOLDER}\Etat_Tonkpi_Liv1_22-03-26_essai.xlsx"
path_etat_voisin_csv = fr"{MAIN_FOLDER}\data_tokpi_19_03_26.csv"
path_voisins_signes_cni = r"C:\Users\L14\Desktop\scripts_python\matches_repr_test.csv"
path_chef_village = r"C:\Users\L14\Desktop\controle_qualite\chef_de_village_avec_cni.csv"

paths_rem = Path(fr"{MAIN_FOLDER}\ALL_CSV\CONTROLE CTB REMARQUES")
paths_dig = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG")
paths_dig.mkdir(parents=True, exist_ok=True)
paths_ctb = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB")
paths_ctb.mkdir(parents=True, exist_ok=True)

out_add_to_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\ADD_TO_CTB")
out_add_to_ctb_folder.mkdir(parents=True, exist_ok=True)
out_no_mask_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\NO_MASK")
out_no_mask_folder.mkdir(parents=True, exist_ok=True)
out_replace_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\REPLACE_CTB")
out_replace_ctb_folder.mkdir(parents=True, exist_ok=True)
out_dig_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG")
out_dig_folder.mkdir(parents=True, exist_ok=True)
out_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB")
out_ctb_folder.mkdir(parents=True, exist_ok=True)
out_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\DIG\UPDATES_FILES")
out_folder.mkdir(parents=True, exist_ok=True)
out_to_ctb_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB\ADD_NEIGHBOR\TO_CTB")
out_to_ctb_folder.mkdir(parents=True, exist_ok=True)
out_ctb_manual_folder = Path(fr"{MAIN_FOLDER}\ACTIONS\CTB\ADD_NEIGHBOR\MANUAL")
out_ctb_manual_folder.mkdir(parents=True, exist_ok=True)

paths_action_dig = paths_dig.glob("*.csv")
paths_action_ctb = paths_ctb.glob("*.csv")
CSVs = paths_dig.glob("*.csv")
files_rem = paths_rem.rglob("*.xlsx")

df_maj_gestionnaire = pd.read_excel(path_maj_gestionnaire, engine='openpyxl')
etat_voisins = pd.read_excel(path_etat_voisin, engine='openpyxl')
#etat_voisins = pd.read_csv(path_etat_voisin_csv)
df_voisins_non_par = pd.read_excel(path_voisins_non_par, engine='openpyxl')
df_voisins_signes_ctrl = pd.read_excel(path_voisins_signes, engine='openpyxl')
df_voisins_pvcl = pd.read_excel(path_voisins_pvcl, engine='openpyxl')
#df_voisins_signes_cni = pd.read_csv(path_voisins_signes_cni, sep=";", encoding="cp1252")
df_voisins_signes_cni = pd.read_csv(path_voisins_signes_cni, sep=";", encoding="utf-8-sig")
df_chef_village = pd.read_csv(path_chef_village)

#controle_voisins = pd.read_excel(path, engine='openpyxl')
df_voisins_signes_ctrl = df_voisins_signes_ctrl[
    df_voisins_signes_ctrl["memberType"].isin([7, "007", 13, "013"])
]
df_maj_gestionnaire.loc[:, "village"] = (
    df_maj_gestionnaire["village"]
      .apply(lambda x: unidecode(x) if pd.notna(x) else x)
      .str.lower()
      .str.strip()
    )

df_maj_gestionnaire.loc[:, "code_par"] = df_maj_gestionnaire.loc[:, "village"] + "-" + df_maj_gestionnaire.loc[:, "numParcelleOF_c"]

#df_maj_gestionnaire.set_index("code_par", inplace=True)

etat_voisins["code_voisin_unique"] = (etat_voisins["indice_unique_voisin"].astype(str).str.strip() + "__" + etat_voisins["num_demande"].astype(str).str.strip())
etat_voisins = etat_voisins[etat_voisins["num_demande"].notna()]
etat_voisins_duplicated = etat_voisins[etat_voisins["code_voisin_unique"].duplicated()]
etat_voisins = etat_voisins.drop_duplicates(subset="code_voisin_unique")
etat_voisins['cod_sp'] = etat_voisins['num_demande'].astype(str).str.split("-").str[0]



df_voisins_signes_ctrl.loc[:, "village"] = (df_voisins_signes_ctrl["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )

df_voisins_pvcl.loc[:, "village"] = (df_voisins_pvcl["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )


df_voisins_pvcl.loc[:, "code_vois"] = (
    df_voisins_pvcl["code"].astype(str).str.strip()
    + "-"
    + df_voisins_pvcl["nameOfNeighbor"].astype(str).str.strip()
)

df_voisins_pvcl = (
    df_voisins_pvcl
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_non_par.loc[:, "village"] = (df_voisins_non_par["village"]
                                        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
                                        .str.lower()
                                        .str.strip()
                                    )
#id	firstDate	secondDate	comityAttestSignatoryCE	nameCVGFR	limitVoiz	nameVoiz	comityAttestSignatoryCVGFR	comityAttestSignatoryPCVGFR	creationDate	active	nameOfPerson_control	cod_vil	code	label	description	village	numParcelleOF	estimatedArea	agent	codeLand



In [9]:
### Generate the file actions_digifor and actions_ctb to update data

#print(etat_voisins_duplicated.shape)
#print(etat_voisins_duplicated["code_voisin_unique"].head(10))

assert etat_voisins["code_voisin_unique"].is_unique, "❌ Le code voisin unique ne l'est pas"
etat_voisins.set_index("code_voisin_unique", drop=False, inplace=True)

def nearest_text_match(target, candidates, target_type='dig', min_score=0.5):
        """
        target : str (voisin DIGIFOR)
        candidates : list[str] (voisins CTB)
        """
        best = None
        unmatched = set(candidates)
        best_score = 0

        for c in candidates:
            if ("Aucun voisin" in target) or ("Aucun voisin" in c):
                    score = text_similarity(target, c)
            elif target_type == "dig":
                #print(c)
                score = text_similarity(target.split('(')[0], c.split(':')[-1].split('(')[0])
            elif target_type == "ctb":
                score = text_similarity(target.split(':')[-1].split('(')[0], c.split('(')[0])
            else:
                score = text_similarity(target, c)

            if score > best_score:
                best = c
                best_score = score
            
        if best_score >= min_score:
            unmatched -= {best}
            return best, unmatched, best_score

        return None, unmatched, best_score

def best_match_nn(digs, ctbs, min_score=0.6):
    if not digs or not ctbs:
        return [], digs.copy(), ctbs.copy()

    matrix = np.zeros((len(digs), len(ctbs)))

    for i, d in enumerate(digs):
        for j, c in enumerate(ctbs):
            if ("Aucun voisin" in d) or ("Aucun voisin" in c):
                score = text_similarity(d, c)
            else:
                score = text_similarity(d.split('(')[0], c.split(':')[-1].split('(')[0])
            matrix[i, j] = score

    row_ind, col_ind = linear_sum_assignment(-matrix)

    matches = []
    used_dig = set()
    used_ctb = set()

    for i, j in zip(row_ind, col_ind):
        score = matrix[i, j]
        if score >= min_score:
            matches.append((digs[i], ctbs[j], score))
            used_dig.add(digs[i])
            used_ctb.add(ctbs[j])

    unmatched_dig = [d for d in digs if d not in used_dig]
    unmatched_ctb = [c for c in ctbs if c not in used_ctb]

    return matches, unmatched_dig, unmatched_ctb

def split_clean(value):
    if pd.isna(value) or value == "RAS":
        return []
    return [v.strip() for v in value.split(",") if v.strip()]

def is_par(v): return re.match(r"^par\d+", v)
def is_riv(v): return re.match(r"^riv\d+", v)
def is_tit(v): return re.match(r"^tit\d+", v)
def is_zone(v): return re.match(r"^zone\d+", v)
def is_det(v): return "det_poly" in v
def is_signe(v): return "(signe)" in v or "(repr)" in v or "(non_par)" in v

def text_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def get_voisin_info(v, row_code):
    code_v = v.split(':')[0] + "__" + row_code
    if code_v in etat_voisins.index:
        r = etat_voisins.loc[code_v]
        code_voisin = r["num_plle_voisin"]
        num_voisin = r["voisin_num_demande"]
        return num_voisin, code_voisin, r["type_voisin"], r["orientation"]
    return None, None, None, None

def compare_names(row, code):
    
    rows_dig = []
    rows_ctb = []

    voisins_dig = split_clean(row["A corriger sur CTB ou Supprimer sur DIGIFOR"])
    voisins_ctb = split_clean(row["A ajouter ou Corriger sur DIGIFOR"])

    taille_dig = len(voisins_dig)
    taille_ctb = len(voisins_ctb)

    def add_row(old, new,code_voisin, code_voisin_plle, type_voisin, position, action, comment):
        rows_dig.append({
            "code": row["code"],
            "code_parcelle": row["numParcelleOF"],
            "village": row["village"],
            "date": row["requestDate"],
            "ancien_nom": old,
            "nouveau_nom": new,
            "code_voisin": code_voisin,
            "code_voisin_plle": code_voisin_plle,
            "type_voisin": type_voisin,
            "position": position,
            "action": action,
            "commentaire": comment
        })
    
    def add_row_ctb(old, new, type_voisin, position, action, comment):
        rows_ctb.append({
            "code": row["code"],
            "code_parcelle": row["numParcelleOF"],
            "village": row["village"],
            "date": row["requestDate"],
            "ancien_nom": old,
            "nouveau_nom": new,
            "type_voisin": type_voisin, 
            "position": position,
            "action": action,
            "commentaire": comment
        })

    
    if voisins_dig == ["Aucun voisin dans le pvcl presence"] or voisins_ctb == ["Aucun voisin dans le ctb"]:
        return [], []
    
    # 1️⃣ DIGIFOR RAS
    elif not voisins_dig and voisins_ctb:
        for v in voisins_ctb:
            code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout depuis CTB (DIGIFOR = RAS)")

    # 2️⃣ CTB RAS
    elif not voisins_ctb and voisins_dig:
        for v in voisins_dig:
            add_row(v, None, "", "", "", "", "DELETE", "Suppression (CTB = RAS)")

    # 3️⃣ 1–1
    elif taille_ctb == taille_dig == 1:
        ctb, dig = voisins_ctb[0], voisins_dig[0]
        code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(ctb, row["code"])
        add_row(None, ctb, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajouter le voisin ctb = (par,zone) 1–1")
        add_row(dig, None, "", "", "", "", "DELETE", "Supprimer le voisin dig, ctb = (par,zone) 1–1")

    # 4️⃣ 1 DIG vs N CTB
    elif taille_dig == 1:
        dig = voisins_dig[0]

        for v in voisins_ctb:
            code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout de voisin all(voisin par,,zone)")
        add_row(dig, None, "", "", "", "", "DELETE", "Suppression ancien voisin (all ctb =par,,zone)")


    # 5️⃣ 1 CTB vs N DIG
    elif taille_ctb == 1:
        ctb = voisins_ctb[0]
        code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(ctb, row["code"])
        add_row(None, ctb, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det)")
        for v in voisins_dig:
            add_row(v, None, "", "", "", "", "DELETE", "Nettoyage DIGIFOR")

                
    # 6️⃣ N–N
    else:
        for v in voisins_ctb:
            code_voisin, code_voisin_plle, type_voisin, position = get_voisin_info(v, row["code"])
            add_row(None, v, code_voisin, code_voisin_plle, type_voisin, position, "ADD", "Ajout de voisin (voisin par ou det ou tit ou zone)")
        for v in voisins_dig:
            add_row(v, None, "", "", "", "", "DELETE", "Nettoyage DIGIFOR all(voisin ctb = par ou det ou tit ou zone)")

    return rows_dig, rows_ctb

for file in files_rem:
    print(file)
    controle_voisins = pd.read_excel(file, engine='openpyxl')
    all_rows_dig = []
    all_rows_ctb = []
    
    for idx, row in controle_voisins.iterrows():
        result_dig, result_ctb = compare_names(row, code=idx)
        all_rows_dig.extend(result_dig)
        all_rows_ctb.extend(result_ctb)

    result_df_dig = pd.DataFrame(all_rows_dig,
                            columns=["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "type_voisin", "code_voisin", "code_voisin_plle", "position", "action", "commentaire"])
    #result_df_dig["id"] = result_df_dig.index
    result_df_ctb = pd.DataFrame(all_rows_ctb,
                            columns=["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "type_voisin", "position", "action", "commentaire"])
    #result_df_ctb["id"] = result_df_ctb.index
    result_df_dig.to_csv(f"{out_dig_folder}/{file.name.split('.')[0]}_actions_digifor.csv",
                    index=False,
                    sep=";",
                    encoding="utf-8-sig")

C:\Users\L14\Downloads\ABH_CSV\ALL_CSV\CONTROLE CTB REMARQUES\ALL_TABLES_EXPORT bossoh (1)_ctb.xlsx


In [ ]:
# --- Le but est de recuperer les signatures, cni dans le village pour les voisins à ajouter dans DIGIFOR ---
def normalize(name):
    return '_'.join(str(name).lower().split()).strip()

update_files_resume = []

df_maj_gestionnaire["requestNumber"] = df_maj_gestionnaire["requestNumber"].astype(str)
df_maj_gestionnaire['fullname'] = df_maj_gestionnaire['fullname'].astype(str).apply(lambda x: unidecode(normalize(str(x))))

df_maj_gestionnaire["code_dem"] = (
        df_maj_gestionnaire["village"] + "-" + df_maj_gestionnaire["fullname"]
        .fillna("")
        .astype(str)
        .str.strip())

df_voisins_signes_cni["code_vois"] = (
    df_voisins_signes_cni["village"].astype(str).str.strip()
    + "-"
    + df_voisins_signes_cni["nom_ctb"].astype(str).str.strip()
)

df_voisins_signes_ctrl["code_num"] = (
    df_voisins_signes_ctrl["codePresence"].astype(str).str.strip()
    + "-"
    + df_voisins_signes_ctrl["nameOfPerson"].astype(str).str.strip()
)

df_voisins_signes_ctrl["code_num_repr"] = (
    df_voisins_signes_ctrl["codePresence"].astype(str).str.strip()
    + "-"
    + df_voisins_signes_ctrl["representant_clean"].astype(str).str.strip()
)

df_voisins_signes_ctrl["code_vois"] = (
    df_voisins_signes_ctrl["village"].astype(str).str.strip()
    + "-"
    + df_voisins_signes_ctrl["nameOfPerson"].astype(str).str.strip()
)

df_voisins_signes_ctrl["code_repr"] = (
    df_voisins_signes_ctrl["village"].astype(str).str.strip()
    + "-"
    + df_voisins_signes_ctrl["representant_clean"].astype(str).str.strip()
)

df_voisins_non_par.loc[:, "code_vois"] = (
    df_voisins_non_par.loc[:, "village"].astype(str).str.strip() 
    + "-" 
    + df_voisins_non_par.loc[:, "nameOfPerson_control"].astype(str).str.strip()
)

df_maj_gestionnaire_dem = df_maj_gestionnaire.drop_duplicates(subset="code_dem", keep="first")
df_maj_gestionnaire_par = df_maj_gestionnaire.drop_duplicates(subset="code_par", keep="first")
df_maj_gestionnaire = df_maj_gestionnaire.drop_duplicates("requestNumber")

df_voisins_signes_ctrl_repr = (df_voisins_signes_ctrl.drop_duplicates(subset="code_repr", keep="first"))
df_voisins_signes_ctrl_vois = (df_voisins_signes_ctrl.drop_duplicates(subset="code_vois", keep="first"))

df_voisins_signes_cni = (
    df_voisins_signes_cni
    .drop_duplicates("code_vois", keep="first")
)

df_voisins_non_par = (
    df_voisins_non_par
    .drop_duplicates("code_vois", keep="first")
)

# ------------------------------------------------
# Traitement des fichiers actions_ctb
# ------------------------------------------------

for path_action_dig in paths_action_dig:
    agent = path_action_dig.stem
    out_folder_ = out_folder / agent.upper().replace("_CTB_ACTIONS_DIGIFOR", "").strip()
    out_folder_.mkdir(parents=True, exist_ok=True)
    #df_action_dig.index = range(len(df_action_dig))

    print("Processing:", path_action_dig)

    df_action_dig = pd.read_csv(path_action_dig, sep=";", encoding="utf-8-sig")

    # --- Nettoyage village
    df_action_dig["village"] = (
        df_action_dig["village"]
        .apply(lambda x: unidecode(x) if pd.notna(x) else x)
        .str.lower()
        .str.strip()
    )

    # --- Masks
    mask_parcelle = df_action_dig["type_voisin"].astype(str) == "voisin_plle"
    mask_riv = (df_action_dig["type_voisin"] == "voisin_riv")
    mask_delete = df_action_dig["action"].isin(["DELETE", "REPLACE_D"])
    

    # --- Nettoyage noms
    nom_nouveau = (
        df_action_dig["nouveau_nom"]
        .astype(str)
        .str.split(":", n=1)
        .str[-1]
        .str.split("(")        
        .str[0]
        .str.strip()    
    )

    df_action_dig.loc[mask_riv, "nom_a_ajouter"] = nom_nouveau
    df_action_dig.loc[mask_riv, "name"] = nom_nouveau
    df_action_dig.loc[mask_parcelle, "name"] = nom_nouveau

    df_action_dig.loc[mask_delete, "name"] = (
        df_action_dig.loc[mask_delete, "ancien_nom"]
        .astype(str)
        .str.split("(")
        .str[0]
        .str.strip()
    )

    # --- Création code_vois
    df_action_dig["code_vois"] = (
        df_action_dig["village"]
        + "-"
        + df_action_dig["name"]
            .fillna("")
            .astype(str)
            .str.split("(")
            .str[0]
            .str.strip()
    )

    df_action_dig["code_num"] = (
        df_action_dig["code"]
        + "-"
        + df_action_dig["name"]
            .fillna("")
            .astype(str)
            .str.split("(")
            .str[0]
            .str.strip()
    )

    # ------------------------------------------------
    # MERGE gestionnaire pour les voisins par
    # ------------------------------------------------
    
    # Harmonisation des types
    df_action_dig["code_voisin"] = df_action_dig["code_voisin"].astype(str)

    # Merge
    df_action_dig["code_par"] = df_action_dig["village"] + "-" + df_action_dig["code_voisin_plle"]
    
    df_action_dig_delete = df_action_dig[mask_delete]
    print("Voisins à supprimer dans DIGIFOR:", df_action_dig_delete.shape[0])
    df_action_dig_add = df_action_dig[~mask_delete]
    print("Voisins à ajouter dans DIGIFOR:", df_action_dig_add.shape[0])

    df_action_dig_add = df_action_dig_add.merge(
        df_maj_gestionnaire[
            ["requestNumber", "identityDocumentNumber_x", "identityDocumentPhoto"]
        ],
        left_on="code_voisin",
        right_on="requestNumber",
        how="left",
        validate="m:1"
    )

    df_action_dig_add.rename(columns={
        "code_voisin": "num_voisin",
        "nom_sig_original": "name_sig_voisin",
        "identityDocumentNumber_x": "cni_voisin",
        "identityDocumentPhoto": "signature_voisin"
    }, inplace=True, errors="ignore")
    
    df_action_dig_add = df_action_dig_add.merge(
        df_maj_gestionnaire_par[["code_par", "requestNumber", "identityDocumentNumber_x", "identityDocumentPhoto"]],
        on="code_par",
        how="left",
        suffixes=("", "_par"),
        validate="many_to_one"
    )

    df_action_dig_add = df_action_dig_add.merge(
        df_maj_gestionnaire_dem[["code_dem", "requestNumber", "identityDocumentNumber_x", "identityDocumentPhoto"]],
        left_on="code_vois",
        right_on="code_dem",
        how="left",
        suffixes=("", "_dem"),
        validate="many_to_one"
    )
    
    # Fallback CNI
    df_action_dig_add["cni_voisin"] = (
        df_action_dig_add["cni_voisin"]
        .combine_first(df_action_dig_add["identityDocumentNumber_x"])
        .combine_first(df_action_dig_add["identityDocumentNumber_x_dem"])
    )

    
    # Fallback signature
    df_action_dig_add["signature_voisin"] = (
        df_action_dig_add["signature_voisin"]
        .combine_first(df_action_dig_add["identityDocumentPhoto"])
        .combine_first(df_action_dig_add["identityDocumentPhoto_dem"])
    )

    df_action_dig_add["num_voisin"] = (
        df_action_dig_add["num_voisin"]
        .combine_first(df_action_dig_add["requestNumber"])
        .combine_first(df_action_dig_add["requestNumber_dem"])
    )


    # ------------------------------------------------
    # MERGE signatures voisins
    # ------------------------------------------------
    df_voisins_signes_ctrl_add = df_voisins_signes_ctrl.drop_duplicates(subset=["code_num"], keep="first")
    
    df_action_dig_add = df_action_dig_add.merge(
        df_voisins_signes_ctrl_add[
            ["code_num", "nameOfPersonOriginal", "numberCNI", "signatoryPhoto", "representant_clean"]
        ],
        on="code_num",
        how="left",
        validate="m:1"
    )

    df_action_dig_delete = df_action_dig_delete.merge(
        df_voisins_signes_ctrl[
            ["code_num", "nameOfPersonOriginal", "numberCNI", "signatoryPhoto", "representant_clean"]
        ],
        on="code_num",
        how="left",
        validate="m:m"
    )

    df_action_dig_add.rename(columns={
        "nameOfPersonOriginal": "name_sig_voisin",
        }, inplace=True, errors="ignore") 
    
    df_action_dig_add["cni_voisin"] = (
        df_action_dig_add["cni_voisin"]
        .combine_first(df_action_dig_add["numberCNI"])
    )
    df_action_dig_add["signature_voisin"] = (
        df_action_dig_add["signature_voisin"]
        .combine_first(df_action_dig_add["signatoryPhoto"])
    )
    
    df_action_dig_delete.rename(columns={
        "nameOfPersonOriginal": "name_sig_voisin",
        "numberCNI": "cni_voisin",
        "signatoryPhoto": "signature_voisin"
        }, inplace=True, errors="ignore") 
    
    # ------------------------------------------------
    # MERGE PVCL ['nameOfPersonOriginal', 'nameOfNeighborOriginal', 'position_ctb']
    # ------------------------------------------------

    df_voisins_signes_ctrl_add = df_voisins_signes_ctrl.drop_duplicates(subset=["code_num_repr"], keep="first")
    df_action_dig_add = df_action_dig_add.merge(
        df_voisins_signes_ctrl_add[
            ["code_num_repr", "nameOfPersonOriginal", "numberCNI", "signatoryPhoto", "representant_clean", "representantOriginal"]
        ],
        left_on="code_num",
        right_on="code_num_repr",
        how="left",
        suffixes=("", "_repr"),
        validate="m:1"
    )

    df_action_dig_delete = df_action_dig_delete.merge(
        df_voisins_signes_ctrl[
            ["code_num_repr", "nameOfPersonOriginal", "numberCNI", "signatoryPhoto", "representant_clean", "representantOriginal"]
        ],
        left_on="code_num",
        right_on="code_num_repr",
        how="left",
        suffixes=("", "_repr"),
        validate="m:m"
    )

    df_action_dig_add["signature_voisin"] = (
        df_action_dig_add["signature_voisin"]
        .combine_first(df_action_dig_add["signatoryPhoto_repr"])
    )

    df_action_dig_add["name_sig_voisin"] = (
        df_action_dig_add["name_sig_voisin"]
        .combine_first(df_action_dig_add["nameOfPersonOriginal"])
    )

    df_action_dig_add["cni_voisin"] = (
        df_action_dig_add["cni_voisin"]
        .combine_first(df_action_dig_add["numberCNI_repr"])
    )

    df_action_dig_delete["signature_voisin"] = (
        df_action_dig_delete["signature_voisin"]
        .combine_first(df_action_dig_delete["signatoryPhoto"])
    )

    df_action_dig_delete["name_sig_voisin"] = (
        df_action_dig_delete["name_sig_voisin"]
        .combine_first(df_action_dig_delete["nameOfPersonOriginal"])
    )

    df_action_dig_delete["cni_voisin"] = (
        df_action_dig_delete["cni_voisin"]
        .combine_first(df_action_dig_delete["numberCNI"])
    )

    df_action_dig_add.drop(
        columns=[
            "nameOfPersonOriginal",
            "numberCNI",
            "signatoryPhoto",
        ],
        errors="ignore",
        inplace=True
    )

    df_action_dig_add = df_action_dig_add.merge(
        df_voisins_signes_cni[
            ["code_vois", "nom_sig_original", "nom_repr", "nom_repr_original", "code_sig", "signatory_photo", "number_cni", "is_representant_match", "qualite_match"]
        ],
        on="code_vois",
        how="left",
        suffixes=("","_dig"),
        validate="m:1"
    )

    df_action_dig_add["name_sig_voisin"] = (
        df_action_dig_add["name_sig_voisin"]
        .combine_first(df_action_dig_add["nom_sig_original"])
    )

    df_action_dig_add["cni_voisin"] = (
        df_action_dig_add["cni_voisin"]
        .combine_first(df_action_dig_add["number_cni"])
    )
    df_action_dig_add["signature_voisin"] = (
        df_action_dig_add["signature_voisin"]
        .combine_first(df_action_dig_add["signatory_photo"])
    )
    # ------------------------------------------------
    # MERGE voisins non par nameCVGFR #id	firstDate	secondDate	comityAttestSignatoryCE	nameCVGFR	limitVoiz	nameVoiz	comityAttestSignatoryCVGFR	comityAttestSignatoryPCVGFR	creationDate	active	nameOfPerson_control	cod_vil	code	label	description	village	numParcelleOF	estimatedArea	agent	codeLand
    # ------------------------------------------------
    
    df_action_dig_add = df_action_dig_add.merge(
        df_voisins_non_par[["code_vois", "code", "firstDate","secondDate","comityAttestSignatoryCE","nameCVGFR","limitVoiz","nameVoiz","comityAttestSignatoryCVGFR","comityAttestSignatoryPCVGFR","creationDate","active"]],
        on="code_vois",
        how="left",
        suffixes=("","_np")
    )
    
    # ------------------------------------------------
    # Nettoyage colonnes
    # ------------------------------------------------

    df_action_dig_add.drop(
        columns=[
            "identityDocumentNumber_x",
            "identityDocumentPhoto",
            "requestNumber",
            "code_par",
            "identityDocumentNumber_x_par",
            "identityDocumentNumber_x_dem",
            "identityDocumentPhoto_par",
            "identityDocumentPhoto_dem",
            "nameOfPersonOriginal",
            "numberCNI",
            "signatoryPhoto",
            "nameOfPersonOriginal_repr",
            "numberCNI_repr",
            "signatoryPhoto_repr"
        ],
        errors="ignore",
        inplace=True
    )


    # ------------------------------------------------
    # DEBUG
    # ------------------------------------------------

    #print("Total actions:", len(df_action_dig))
    #{print("CNI trouvées:", df_action_dig["cni_voisin"].notna().sum())

    ## Generate the files add_data.csv, replace_data.csv, delete_data.csv from actions_digifor.csv
    
    print(df_action_dig_add.shape[0], "voisins à ajouter dans DIGIFOR")
    ### Generate add_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_ajouter, type_voisin, num_voisin, cni_voisin, signature_voisin, a_signe)
    
    mask_zone = df_action_dig_add["nouveau_nom"].astype(str).str.contains("zone|terre|communautaire|foret|sacree", case=False, na=False)
    is_voisin_plle = (df_action_dig_add["type_voisin"].isin(["voisin_plle"]) &
        (df_action_dig_add["signature_voisin"].fillna("") != ""))
    
    is_non_par = df_action_dig_add["nameCVGFR"].fillna("").str.strip() != ""
    is_voisin_riv = (
        (df_action_dig_add["type_voisin"] == "voisin_riv") &
        (df_action_dig_add["signature_voisin"].fillna("") != "")
    )

    mask_voisin_plle = (
        (is_voisin_plle) |
        (is_voisin_riv)
    )
    #print(mask_voisin_plle.sum(), "voisins de parcelle à ajouter dans DIGIFOR")
    #print(is_non_par.sum(), "voisins non participants à ajouter dans DIGIFOR")
    #print(mask_zone.sum(), "zones à ajouter dans DIGIFOR")
    
    df_action_dig_add["nom_a_ajouter"] = (
        df_action_dig_add["nouveau_nom"].astype(str).str.split(":").str[1].str.replace("_"," ").str.strip()
    )
    df_action_dig_add["code_uniq"] = (
        df_action_dig_add["nouveau_nom"].astype(str).str.split(":").str[0].str.strip()
    )
    
    df_action_dig_plle = df_action_dig_add[mask_voisin_plle ].drop_duplicates(subset=["code", "code_parcelle", "village", "date","nom_a_ajouter", "code_uniq", "cni_voisin", "signature_voisin"], keep="first")
    df_action_dig_plle[["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_ajouter", "is_representant_match", "qualite_match", "name_sig_voisin", "nom_repr", "nom_repr_original", "type_voisin", "num_voisin", "cni_voisin", "signature_voisin"]].to_csv(f"{out_folder_}/ajout_voisins_fiche_presence.csv", sep=",", encoding="utf-8-sig")

    ### Generate delete_data.csv (code, code_parcelle, village, date, code_uniq, nom_a_supprimer, type_voisin, a_signe)

    df_action_dig_delete["nom_a_supprimer"] = df_action_dig_delete["name"].astype(str).str.split("(").str[0].str.replace("_"," ").str.strip()

    df_action_dig_delete[["code", "code_parcelle", "village", "date", "nom_a_supprimer","name_sig_voisin", "representant_clean", "representantOriginal", "cni_voisin", "type_voisin"]].to_csv(f"{out_folder_}/suppression_voisins_fiche_presence.csv", sep=",", encoding="utf-8-sig")

    ### Generate non_par_data.csv ("code", "nameVoiz", "nameCVGFR", "cod_vil","firstDate","secondDate","comityAttestSignatoryCE","nameCVGFR","limitVoiz","nameVoiz","comityAttestSignatoryCVGFR","comityAttestSignatoryPCVGFR","creationDate","active")

    df_action_dig_add.loc[is_non_par, ["code", "nameVoiz", "nameCVGFR", "firstDate", "secondDate", "comityAttestSignatoryCE","nameCVGFR","limitVoiz","nameVoiz","comityAttestSignatoryCVGFR","comityAttestSignatoryPCVGFR","creationDate","active"]].to_csv(f"{out_folder_}/ajout_voisins_pv_non_participation.csv", sep=",", encoding="utf-8-sig")

    
    # ------------------------------------------------
    # Recupération signature et CNI depuis chef de village
    # ------------------------------------------------
    df_action_dig_add["cod_vil"] = df_action_dig_add['code'].str.split('-').str[:2].str.join('-')
    df_action_dig_zone = df_action_dig_add[mask_zone]
    df_chef_village.drop_duplicates(subset=["cod_vil"], keep="first", inplace=True)
    df_action_dig_zone = df_action_dig_zone.merge(
        df_chef_village[["cod_vil", "identityDocumentNumber", "identityDocumentPhoto", "nameInterviewedPerson"]],
        left_on="cod_vil",
        right_on="cod_vil",
        how="left",
        validate="m:1"
    )

    df_action_dig_zone.rename(
        columns={"identityDocumentNumber": "cni_chef_village", "identityDocumentPhoto": "signature_chef_village", "nameInterviewedPerson": "nom_chef_village"},
        inplace=True
    )
    df_action_dig_zone["nouveau_nom"] = df_action_dig_zone["nouveau_nom"].astype(str).str.split(":").str[-1].str.replace("_"," ").str.strip()
    
    df_action_dig_zone.drop_duplicates(subset=["code"], keep='first', inplace=True)
    df_action_dig_zone[["code", "code_parcelle", "village", "date", "ancien_nom", "nouveau_nom", "nom_chef_village", "type_voisin", "cni_chef_village", "signature_chef_village"]].to_csv(f"{out_folder_}/ajout_zones_exclus_fiche_presence.csv", sep=",", encoding="utf-8-sig")
    
    no_mask = ~ (mask_voisin_plle | is_non_par | mask_zone)
    
    df_action_dig_add.loc[no_mask, ["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_ajouter", "code_voisin_plle", "is_representant_match", "qualite_match", "name_sig_voisin", "nom_repr", "nom_repr_original", "type_voisin", "num_voisin", "cni_voisin", "signature_voisin"]].to_csv(f"{out_folder_}/nomask_data.csv", sep=";", encoding="utf-8-sig") 
    
    df_action_dig_add_no_mask = df_action_dig_add[no_mask]
    df_action_dig_add_no_mask_plle = df_action_dig_add_no_mask[df_action_dig_add_no_mask['code_uniq'].astype(str).str.startswith("par")]
    df_action_dig_add_no_mask_riv = df_action_dig_add_no_mask[~(df_action_dig_add_no_mask['code_uniq'].astype(str).str.startswith("par"))]

    df_action_dig_add_no_mask_plle[["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_ajouter", "code_voisin_plle", "is_representant_match", "qualite_match", "name_sig_voisin", "nom_repr", "nom_repr_original", "type_voisin", "num_voisin", "cni_voisin", "signature_voisin"]].to_csv(f"{out_no_mask_folder}/{agent}_nomask_data_plle.csv", sep=";", encoding="utf-8-sig") 
    df_action_dig_add_no_mask_riv[["code", "code_parcelle", "village", "date", "code_uniq", "nom_a_ajouter", "code_voisin_plle", "is_representant_match", "qualite_match", "name_sig_voisin", "nom_repr", "nom_repr_original", "type_voisin", "num_voisin", "cni_voisin", "signature_voisin"]].to_csv(f"{out_no_mask_folder}/{agent}_nomask_data_riv.csv", sep=";", encoding="utf-8-sig") 


    update_files_resume.append({
        "Agent": agent,
        "Total lignes" : len(df_action_dig),
        "ADD" : mask_voisin_plle.sum() + mask_zone.sum(),
        "DELETE": mask_delete.sum(),
        "NON_PAR": is_non_par.sum(),
        "NO_MASK" : no_mask.sum(),
        "Somme totale" :
        mask_voisin_plle.sum()
        + mask_delete.sum()
        + is_non_par.sum()
        + no_mask.sum()
    })


df_update_files = pd.DataFrame(update_files_resume,
                               columns=["Agent", "Total lignes", "ADD", "REPLACE", "DELETE", "NON_PAR", "NO_MASK", "Somme totale"])

df_update_files.to_csv(f"resume_data.csv", sep=";", encoding="utf-8-sig")

Processing: C:\Users\L14\Downloads\ABH_CSV\ACTIONS\DIG\ALL_TABLES_EXPORT bossoh (1)_ctb_actions_digifor.csv
Voisins à supprimer dans DIGIFOR: 194
Voisins à ajouter dans DIGIFOR: 238
238 voisins à ajouter dans DIGIFOR


In [11]:
### Generate the files add_data.csv, replace_data.csv, delete_data.csv from actions_digifor.csv
update_files_resume = []
    